<a href="https://colab.research.google.com/github/tahzibultamim/PythonProjects/blob/sM/6_Data_Preprocessing_and_Pipeline-DS3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **0. Setup and Imports**

In [74]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [75]:
from google.colab import drive
drive.mount('/content/drive')
import warnings
warnings.filterwarnings('ignore')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# 📑 Table of Contents

0. Setup and Imports
1. Loading and Exploring the Dataset
2. Train-Test Split
3. Feature Scaling
    - 3.1 Why Scale?
    - 3.2 StandardScaler
    - 3.3 MinMaxScaler (Min-Max Normalization)
4. Encoding Categorical Data
    - 4.1 Label Encoding (For Binary/Two Categories)
    - 4.2 One-Hot Encoding (For Multiple Categories)
5. Train a Simple Model
6. Cross-Validation
7. Complete Pipeline (All Steps Together)
8. Assignment 1: Agriculture Dataset (Regression Practice)
9. Assignment 2: Housing Dataset (Classification Practice)

# **1. Loading and Exploring the Dataset**

In [76]:
df = pd.read_csv('/content/drive/MyDrive/datasets/diabetes.csv')  # No header in raw CSV

In [77]:
# Basic exploration
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

print("\nDataset Info:")
print(df.info())

print("\nBasic Statistics:")
print(df.describe())

print("\nCheck for missing values:")
print(df.isnull().sum())

print("\nTarget variable distribution:")
print(df['Outcome'].value_counts())

Dataset Shape: (768, 9)

First 5 rows:
   Pregnancies  Glucose  BloodPressure  SkinThickness  Insulin   BMI  \
0            6      148             72             35        0  33.6   
1            1       85             66             29        0  26.6   
2            8      183             64              0        0  23.3   
3            1       89             66             23       94  28.1   
4            0      137             40             35      168  43.1   

   DiabetesPedigreeFunction  Age  Outcome  
0                     0.627   50        1  
1                     0.351   31        0  
2                     0.672   32        1  
3                     0.167   21        0  
4                     2.288   33        1  

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 

# **2. Train-Test Split**

In [78]:
# Separate input (X) and output (y)
X = df.drop('Outcome', axis=1)  # Everything except Outcome
y = df['Outcome']                # Only Outcome

print("X has all patient information (8 columns)")
print("y has diabetes yes/no (1 column)")

# Split: 80% for training, 20% for testing
# stratify=y keeps the same Outcome (yes/no) ratio in both train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,    # 20% for test
    stratify=y,       # keep the same Outcome ratio in train and test
    random_state=20   # Same result every time
)

print(f"\nWe will train with: {len(X_train)} patients")
print(f"We will test with: {len(X_test)} patients")



X has all patient information (8 columns)
y has diabetes yes/no (1 column)

We will train with: 614 patients
We will test with: 154 patients


# **3. Feature Scaling**
## **3.1 Why Scale?**

In [79]:
# Problem: Some numbers are big, some are small
print("Before scaling - different ranges:")
print(f"Age: smallest={X_train['Age'].min()}, biggest={X_train['Age'].max()}")
print(f"Insulin: smallest={X_train['Insulin'].min()}, biggest={X_train['Insulin'].max()}")
print("\nAge goes from 21 to 81 (small range)")
print("Insulin goes from 0 to 846 (big range!)")
print("This confuses the model!")

Before scaling - different ranges:
Age: smallest=21, biggest=81
Insulin: smallest=0, biggest=846

Age goes from 21 to 81 (small range)
Insulin goes from 0 to 846 (big range!)
This confuses the model!


## **3.2 StandardScaler**

In [80]:
# Create scaler
scaler = StandardScaler()

# Learn from training data and scale it
X_train_scaled = scaler.fit_transform(X_train)

# Scale test data (just transform, don't fit)
X_test_scaled = scaler.transform(X_test)

print("\nAfter scaling:")
print("All features now have similar range!")
print(f"Example - First patient's age before scaling: {X_train.values[0][7]}")
print(f"Example - First patient's age after scaling: {X_train_scaled[0][7]:.2f}")


After scaling:
All features now have similar range!
Example - First patient's age before scaling: 25.0
Example - First patient's age after scaling: -0.69


## **3.3 MinMaxScaler (Min-Max Normalization)**

In [81]:
from sklearn.preprocessing import MinMaxScaler

# Create MinMaxScaler
minmax_scaler = MinMaxScaler()

# Learn from training data and scale it
X_train_minmax = minmax_scaler.fit_transform(X_train)

# Scale test data
X_test_minmax = minmax_scaler.transform(X_test)

print("\nAfter MinMaxScaler:")
print("All features now between 0 and 1!")
print(f"Example - First patient's age before scaling: {X_train.values[0][7]}")
print(f"Example - First patient's age after scaling: {X_train_minmax[0][7]:.2f}")

# Let's verify MinMaxScaler worked
print("\nChecking MinMaxScaler:")
print(f"  Minimum value: {X_train_minmax.min():.1f} (should be 0)")
print(f"  Maximum value: {X_train_minmax.max():.1f} (should be 1)")


After MinMaxScaler:
All features now between 0 and 1!
Example - First patient's age before scaling: 25.0
Example - First patient's age after scaling: 0.07

Checking MinMaxScaler:
  Minimum value: 0.0 (should be 0)
  Maximum value: 1.0 (should be 1)


# **4. Encoding Categorical Data**

In [82]:
# Let's create a sample dataset with text data
sample_data = pd.DataFrame({
    'age': [25, 30, 35, 40],
    'gender': ['male', 'female', 'male', 'female'],
    'smoker': ['yes', 'no', 'yes', 'no'],
    'disease': [1, 0, 1, 0]
})

print("Original data with text:")
print(sample_data)

Original data with text:
   age  gender smoker  disease
0   25    male    yes        1
1   30  female     no        0
2   35    male    yes        1
3   40  female     no        0


## **4.1 Label Encoding (For Binary/Two Categories)**

In [83]:
from sklearn.preprocessing import LabelEncoder

# For yes/no or male/female (2 categories only)
label_encoder = LabelEncoder()

# Convert 'smoker' column
sample_data['smoker_encoded'] = label_encoder.fit_transform(sample_data['smoker'])
print("\nAfter Label Encoding 'smoker':")
print(sample_data[['smoker', 'smoker_encoded']])
# no = 0, yes = 1

# Convert 'gender' column
sample_data['gender_encoded'] = label_encoder.fit_transform(sample_data['gender'])
print("\nAfter Label Encoding 'gender':")
print(sample_data[['gender', 'gender_encoded']])
# female = 0, male = 1


After Label Encoding 'smoker':
  smoker  smoker_encoded
0    yes               1
1     no               0
2    yes               1
3     no               0

After Label Encoding 'gender':
   gender  gender_encoded
0    male               1
1  female               0
2    male               1
3  female               0


## **4.2 One-Hot Encoding (For Multiple Categories)**

In [84]:
# Let's add a column with multiple categories
sample_data['city'] = ['Delhi', 'Mumbai', 'Delhi', 'Kolkata']

# One-Hot Encoding using pandas
city_encoded = pd.get_dummies(sample_data['city'], prefix='city')
print("\nOne-Hot Encoding for 'city':")
print(city_encoded)
# Creates separate columns for each city

# Add to original data
sample_data = pd.concat([sample_data, city_encoded], axis=1)
print("\nFinal data with all encodings:")
print(sample_data)


One-Hot Encoding for 'city':
   city_Delhi  city_Kolkata  city_Mumbai
0        True         False        False
1       False         False         True
2        True         False        False
3       False          True        False

Final data with all encodings:
   age  gender smoker  disease  smoker_encoded  gender_encoded     city  \
0   25    male    yes        1               1               1    Delhi   
1   30  female     no        0               0               0   Mumbai   
2   35    male    yes        1               1               1    Delhi   
3   40  female     no        0               0               0  Kolkata   

   city_Delhi  city_Kolkata  city_Mumbai  
0        True         False        False  
1       False         False         True  
2        True         False        False  
3       False          True        False  


# **5. Train a Simple Model**

In [85]:
from sklearn.linear_model import LogisticRegression

# Create model
model = LogisticRegression(max_iter=1000)

# Train the model
model.fit(X_train_scaled, y_train)
print("✅ Model trained!")

# Check accuracy on training data
train_score = model.score(X_train_scaled, y_train)
print(f"Training accuracy: {train_score:.1%}")

# Check accuracy on test data
test_score = model.score(X_test_scaled, y_test)
print(f"Test accuracy: {test_score:.1%}")

✅ Model trained!
Training accuracy: 77.5%
Test accuracy: 81.2%


# **6. Cross-Validation**

In [86]:
from sklearn.model_selection import cross_val_score

# Do 5-fold cross-validation
scores = cross_val_score(
    model,           # Our model
    X_train_scaled,  # Training data
    y_train,         # Training labels
    cv=5            # 5 mini-tests
)

print("5 mini-test scores:")
for i in range(5):
    print(f"  Test {i+1}: {scores[i]:.1%}")

print(f"\nAverage: {scores.mean():.1%}")
print(f"This means our model is {scores.mean():.1%} accurate!")

5 mini-test scores:
  Test 1: 78.0%
  Test 2: 71.5%
  Test 3: 74.0%
  Test 4: 78.9%
  Test 5: 77.0%

Average: 75.9%
This means our model is 75.9% accurate!


# **7. Complete Pipeline (All Steps Together)**

In [87]:
# Complete simple pipeline
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# 1. Load data
df = pd.read_csv('/content/drive/MyDrive/datasets/diabetes.csv')
print(f"Loaded {len(df)} patients")

# 2. Split X and y
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# 3. Train-test split (stratify=y keeps the same Outcome ratio in both sets)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f"Training: {len(X_train)}, Testing: {len(X_test)}")
print(f"Outcome ratio - train: {y_train.mean():.1%}, test: {y_test.mean():.1%}")

# 4. Scale data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
print("Data scaled")

# 5. Train model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)
print("Model trained")

# 6. Check accuracy
accuracy = model.score(X_test_scaled, y_test)
print(f"Accuracy: {accuracy:.1%}")

Loaded 768 patients
Training: 614, Testing: 154
Outcome ratio - train: 34.9%, test: 35.1%
Data scaled
Model trained
Accuracy: 71.4%


# **8. Assignment 1: Housing Dataset (Classification Practice)**


In [88]:
# ASSIGNMENT 1 — YOUR CODE HERE
# Follow the numbered steps from the markdown cell above.

# 1. Load and explore the dataset

import pandas as pd

# 1. Load the dataset
file_path = '/content/drive/MyDrive/datasets/housing.csv'
df = pd.read_csv(file_path)

# Explore the dataset
print("=== First 5 Rows ===")
display(df.head())

print("\n=== Dataset Info ===")
df.info()

print(f"\nShape: {df.shape[0]} rows, {df.shape[1]} columns")

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== Target Column Distribution ('airconditioning') ===")
print(df['airconditioning'].value_counts())


# 2. Separate X (features) and y (target = 'airconditioning')

X = df.drop(columns=['airconditioning'])
y = df['airconditioning']


# 3. Label Encode: mainroad, guestroom, basement, hotwaterheating, prefarea, airconditioning

from sklearn.preprocessing import LabelEncoder

# ৩.১ LabelEncoder-এর অবজেক্ট তৈরি
le = LabelEncoder()

# ৩.২ X-এর বাইনারি কলামগুলো এনকোড করা
binary_cols = ['mainroad', 'guestroom', 'basement', 'hotwaterheating', 'prefarea']

for col in binary_cols:
    X[col] = le.fit_transform(X[col])

# ৩.৩ টার্গেট ভ্যারিয়েবল y ('airconditioning') এনকোড করা
y = le.fit_transform(y)

# এনকোডিং ঠিকমতো হয়েছে কিনা চেক করা
print("X-এর প্রথম ৩টি সারি (এনকোড করার পর):")
display(X[binary_cols].head(3))

print("\ny-এর প্রথম ৫টি মান:")
print(y[:5])

# 4. One-Hot Encode: furnishingstatus
# 4. One-Hot Encode: furnishingstatus
X = pd.get_dummies(X, columns=['furnishingstatus'], drop_first=True, dtype=int)

# ফলাফল চেক করে দেখার জন্য
print("এনকোডিংয়ের পর X-এর নতুন কলামসমূহ:")
print(X.columns)

print("\nনতুন তৈরি হওয়া কলামগুলোর প্রথম ৩টি সারি:")
display(X.filter(like='furnishingstatus').head(3))


# 5. Train-test split (80% train, 20% test) — remember stratify=y
from sklearn.model_selection import train_test_split

# 5. Train-test split (80% train, 20% test) — remember stratify=y
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# ডেটার সাইজ চেক করার জন্য
print(f"X_train shape: {X_train.shape} | y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape} | y_test shape:  {y_test.shape}")


# 6. Scale the numeric features with StandardScaler
from sklearn.preprocessing import StandardScaler

# ৬.১ যে সংখ্যাসূচক কলামগুলো স্কেল করতে হবে তার তালিকা
numeric_features = ['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'parking']

# ৬.২ StandardScaler-এর একটি অবজেক্ট তৈরি
scaler = StandardScaler()

# ৬.৩ ট্রেনিং ডেটাতে fit এবং transform করা
X_train[numeric_features] = scaler.fit_transform(X_train[numeric_features])

# ৬.৪ টেস্ট ডেটাতে শুধুমাত্র transform করা (Data Leakage এড়াতে)
X_test[numeric_features] = scaler.transform(X_test[numeric_features])

# স্কেল ঠিকমতো হলো কিনা যাচাই করতে প্রথম ৩টি সারি দেখা
display(X_train[numeric_features].head(3))


# 7. Run: train a LogisticRegression model apply cross validation (5,10)and print the accuracy

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
# ১. টেস্ট সেটের অ্যাকুরেসি শতাংশে
print(f"=== Holdout Test Accuracy ===")
print(f"Accuracy: {test_acc * 100:.2f}%")
# অথবা: print(f"Accuracy: {test_acc:.2%}")

# ২. Cross-Validation ফলাফল শতাংশে
print("\n=== Cross-Validation Results ===")
for k in [5, 10]:
    skf = StratifiedKFold(n_splits=k, shuffle=True, random_state=42)
    scores = cross_val_score(cv_pipeline, X, y, cv=skf, scoring='accuracy')

    # প্রতিটি ফোল্ডকে শতাংশে রূপান্তর
    fold_pct = [f"{s * 100:.2f}%" for s in scores]

    print(f"\n>> {k}-Fold Cross-Validation:")
    print(f"Fold Accuracies: {fold_pct}")
    print(f"Mean Accuracy:   {scores.mean() * 100:.2f}%")
    print(f"Std Deviation:   {scores.std() * 100:.2f}%")




=== First 5 Rows ===


,price,area,bedrooms,bathrooms,stories,mainroad,guestroom,basement,hotwaterheating,airconditioning,parking,prefarea,furnishingstatus
0,13300000,7420,4,2,3,yes,no,no,no,yes,2,yes,furnished
1,12250000,8960,4,4,4,yes,no,no,no,yes,3,no,furnished
2,12250000,9960,3,2,2,yes,no,yes,no,no,2,yes,semi-furnished
3,12215000,7500,4,2,2,yes,no,yes,no,yes,3,yes,furnished
4,11410000,7420,4,1,2,yes,yes,yes,no,yes,2,no,furnished



=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 545 entries, 0 to 544
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   price             545 non-null    int64 
 1   area              545 non-null    int64 
 2   bedrooms          545 non-null    int64 
 3   bathrooms         545 non-null    int64 
 4   stories           545 non-null    int64 
 5   mainroad          545 non-null    object
 6   guestroom         545 non-null    object
 7   basement          545 non-null    object
 8   hotwaterheating   545 non-null    object
 9   airconditioning   545 non-null    object
 10  parking           545 non-null    int64 
 11  prefarea          545 non-null    object
 12  furnishingstatus  545 non-null    object
dtypes: int64(6), object(7)
memory usage: 55.5+ KB

Shape: 545 rows, 13 columns

=== Missing Values ===
price               0
area                0
bedrooms            0
bathrooms    

,mainroad,guestroom,basement,hotwaterheating,prefarea
0,1,0,0,0,1
1,1,0,0,0,0
2,1,0,1,0,1



y-এর প্রথম ৫টি মান:
[1 1 0 1 1]
এনকোডিংয়ের পর X-এর নতুন কলামসমূহ:
Index(['price', 'area', 'bedrooms', 'bathrooms', 'stories', 'mainroad',
       'guestroom', 'basement', 'hotwaterheating', 'parking', 'prefarea',
       'furnishingstatus_semi-furnished', 'furnishingstatus_unfurnished'],
      dtype='object')

নতুন তৈরি হওয়া কলামগুলোর প্রথম ৩টি সারি:


,furnishingstatus_semi-furnished,furnishingstatus_unfurnished
0,0,0
1,0,0
2,1,0


X_train shape: (436, 13) | y_train shape: (436,)
X_test shape:  (109, 13) | y_test shape:  (109,)


,price,area,bedrooms,bathrooms,stories,parking
120,0.617469,0.655716,0.061047,-0.562857,-0.934237,1.467701
274,-0.243478,0.613133,1.461920,-0.562857,0.253306,-0.846648
268,-0.221018,-0.096581,1.461920,-0.562857,0.253306,-0.846648


=== Holdout Test Accuracy ===
Accuracy: 75.23%

=== Cross-Validation Results ===

>> 5-Fold Cross-Validation:
Fold Accuracies: ['77.06%', '68.81%', '77.06%', '77.98%', '79.82%']
Mean Accuracy:   76.15%
Std Deviation:   3.80%

>> 10-Fold Cross-Validation:
Fold Accuracies: ['78.18%', '74.55%', '69.09%', '70.91%', '76.36%', '79.63%', '79.63%', '75.93%', '83.33%', '75.93%']
Mean Accuracy:   76.35%
Std Deviation:   3.99%
